In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.functions import current_date, months_between, floor
import time

In [5]:
spark = SparkSession.builder \
    .appName("Spark_Lab") \
    .master("spark://spark-master:7077") \
    .config("spark.ui.port", "4042") \
    .config("spark.executor.instances","2") \
    .config("spark.executor.core","1") \
    .config("spark.executor.memory","1g") \
    .config("spark.cores.max", "2") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/20 14:14:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [6]:
sales_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .load("/opt/spark/work-dir/hr_employees.csv")

In [9]:
df_clean = sales_df \
    .withColumn("salary", col("salary").cast("double")) \
    .withColumn("age", col("age").cast("int")) \
    .withColumn("join_date", to_date(col("join_date"), "yyyy-MM-dd")) \
    .withColumn("performance_score", col("performance_score").cast("double")) \
    .withColumn("employment_status", lower(trim(col("employment_status")))) \
    .fillna({"dept": "Unknown"})

In [10]:
df_clean.show()

[Stage 1:>                                                          (0 + 1) / 1]

+------+------------------+----------+------+----+----------+-----------------+----------+-----------------+
|emp_id|              name|      dept|salary| age| join_date|performance_score|manager_id|employment_status|
+------+------------------+----------+------+----+----------+-----------------+----------+-----------------+
|     1|   Jennifer Acosta|        IT|2824.0|  29|2025-08-24|             NULL|         6|         inactive|
|     2|        Jenna Mayo| Marketing|4811.0|  55|2024-07-26|             91.0|        49|         inactive|
|     3|    Kristin Haynes|   Finance|7924.0|  42|2023-10-17|             94.0|        17|           active|
|     4|    Jessica Murray| Marketing|8527.0|  26|2023-08-03|             NULL|        43|         inactive|
|     5|     Denise Campos|        IT|5741.0|  45|2023-08-16|             62.0|        18|           active|
|     6|       Randy Smith| Marketing|3803.0|  45|2025-10-11|             60.0|        50|           active|
|     7|      Paula

In [13]:
df_clean.select("employment_status").distinct().show()

+-----------------+
|employment_status|
+-----------------+
|           active|
|         inactive|
+-----------------+



In [ ]:
exp_df = active_em_df.withColumn(
    "years_experience",
    floor(months_between(current_date(), col("join_date")) / 12)
)